# Auto MPG — Linear Regression, Feature Scaling & Encoding
## Project 7

Predict `mpg` using Linear Regression while demonstrating numerical scaling,
categorical encoding, missing-value imputation, evaluation and cross-validation.

**Dataset note:** The included 398-row CSV is a reproducible practice dataset with
an Auto MPG-style schema. It is not claimed to be the official UCI file.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,KFold,cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

BASE=Path.cwd()
if not (BASE/'dataset').exists(): BASE=BASE.parent
DATA=BASE/'dataset'/'auto_mpg.csv'; OUT=BASE/'outputs'; OUT.mkdir(exist_ok=True)
df=pd.read_csv(DATA)
display(df.head()); print('Shape:',df.shape)

## 1. Data Quality

In [ ]:
display(df.info())
display(df.describe(include='all').T)
display(pd.DataFrame({'Missing_Count':df.isna().sum(),'Missing_Percent':df.isna().mean()*100}))

## 2. Exploratory Data Analysis

In [ ]:
for x,title in [('mpg','MPG Distribution'),('horsepower','Horsepower vs MPG'),('weight','Weight vs MPG')]:
    plt.figure(figsize=(7,5))
    if x=='mpg': plt.hist(df[x],bins=25); plt.xlabel('MPG'); plt.ylabel('Frequency')
    else: plt.scatter(df[x],df.mpg,alpha=.5); plt.xlabel(x); plt.ylabel('MPG')
    plt.title(title); plt.show()

## 3. Feature Selection and Train/Test Split

In [ ]:
num=['cylinders','displacement','horsepower','weight','acceleration','model_year']
cat=['origin','make']
X=df[num+cat]; y=df['mpg']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42)
print('Train:',X_train.shape,'Test:',X_test.shape)

## 4. Preprocessing Pipeline

**Numerical:** median imputation → StandardScaler

**Categorical:** most-frequent imputation → OneHotEncoder

The preprocessing is inside a `ColumnTransformer` and `Pipeline`, so the same transformations
are applied consistently during training, testing and cross-validation.

In [ ]:
num_pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
cat_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',drop='first'))])
pre=ColumnTransformer([('num',num_pipe,num),('cat',cat_pipe,cat)])
model=Pipeline([('preprocessor',pre),('regressor',LinearRegression())])
model.fit(X_train,y_train)
pred=model.predict(X_test)

## 5. Regression Metrics

In [ ]:
metrics=pd.DataFrame([{
    'MAE':mean_absolute_error(y_test,pred),
    'MSE':mean_squared_error(y_test,pred),
    'RMSE':mean_squared_error(y_test,pred)**.5,
    'R2':r2_score(y_test,pred)
}])
display(metrics)
metrics.to_csv(OUT/'linear_regression_metrics.csv',index=False)

## 6. Actual vs Predicted

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(y_test,pred,alpha=.7)
plt.plot([y_test.min(),y_test.max()],[y_test.min(),y_test.max()],'--')
plt.title('Actual vs Predicted MPG'); plt.xlabel('Actual MPG'); plt.ylabel('Predicted MPG'); plt.show()

## 7. Residual Analysis

In [ ]:
res=y_test-pred
plt.figure(figsize=(7,5)); plt.scatter(pred,res,alpha=.7); plt.axhline(0,linestyle='--')
plt.title('Residuals vs Predicted MPG'); plt.xlabel('Predicted MPG'); plt.ylabel('Residual'); plt.show()
predictions=df.loc[X_test.index,['mpg']].copy()
predictions['Predicted_MPG']=pred; predictions['Residual']=res
display(predictions.head()); predictions.to_csv(OUT/'test_predictions.csv',index=False)

## 8. Model Coefficients

In [ ]:
names=model.named_steps['preprocessor'].get_feature_names_out()
coef_df=pd.DataFrame({'Feature':names,'Coefficient':model.named_steps['regressor'].coef_})
coef_df['Absolute_Coefficient']=coef_df.Coefficient.abs()
display(coef_df.sort_values('Absolute_Coefficient',ascending=False).head(20))
coef_df.to_csv(OUT/'model_coefficients.csv',index=False)

### Interpretation

Numerical features are standardised, so their coefficients describe the effect of a
one-standard-deviation increase, holding other variables constant. Categorical coefficients
are relative to the dropped reference category.

## 9. Five-Fold Cross-Validation

In [ ]:
cv=KFold(n_splits=5,shuffle=True,random_state=42)
r=cross_validate(model,X,y,cv=cv,scoring=['neg_mean_absolute_error','neg_root_mean_squared_error','r2'],n_jobs=-1)
cv_summary=pd.DataFrame({
 'Metric':['MAE','RMSE','R2'],
 'Mean':[-r['test_neg_mean_absolute_error'].mean(),-r['test_neg_root_mean_squared_error'].mean(),r['test_r2'].mean()],
 'Std':[r['test_neg_mean_absolute_error'].std(),r['test_neg_root_mean_squared_error'].std(),r['test_r2'].std()]
})
display(cv_summary); cv_summary.to_csv(OUT/'cross_validation_results.csv',index=False)

## 10. Conclusion

**Pipeline:** Auto MPG data → EDA → missing-value handling → scaling/encoding →
Linear Regression → MAE/MSE/RMSE/R² → residual analysis → coefficient interpretation →
5-fold cross-validation.

The project demonstrates why preprocessing should be encapsulated in a reusable ML pipeline.